In [1]:
import os 
import sys 
import ipynbname
import fnmatch
from pathlib import Path

import logging
from dotenv import load_dotenv
load_dotenv()

python_dir=os.getenv('python_dir')

if python_dir not in sys.path: 
    print(f"adding {python_dir} to sys path")
    sys.path.append(python_dir)

from common_utils import setup_logger
log_dir=os.getenv('log_dir')
log_file=os.path.join(log_dir, f"{ipynbname.name()}.log")
setup_logger('INFO',log_file)

logging.info(f"start  {ipynbname.name()}")

adding /Users/feng/workspace/aimpro/python to sys path


In [2]:
from pathlib import Path 
tmp_dir=os.getenv('tmp_dir')
workspace_dir=os.path.join(tmp_dir, "aimpro",f"{os.getpid()}")
Path(workspace_dir).mkdir(parents=True,exist_ok=True)
workspace_dir
#workspace_dir="/Users/feng/workspace/runtime/tmp/aimpro/29403"

'/Users/feng/workspace/runtime/tmp/aimpro/1637'

In [3]:
video_file_ext="MP4"
extract_fps=3
extract_img_fmt="jpg"
detection_chunk_size=60

In [4]:
import pyperclip
raw_video_dir=r"/Users/feng/workspace/data/bball_video"
left_video_dir=os.path.join(raw_video_dir, 'left')
right_video_dir=os.path.join(raw_video_dir, 'right')

def file_sort_key(f):
    name=os.path.basename(f)
    name=name.split('.')[0].split('_')[-1]
    return int(name)

def gen_concat_cmd(video_dir, concat_file_name): 
    files=fnmatch.filter(os.listdir(video_dir), f"*.{video_file_ext}")
    files=[os.path.join(video_dir, f) for f in files ]
    files=sorted(files, key=file_sort_key)
    list_file_name=os.path.join(workspace_dir, f"{concat_file_name}.list.txt")
    with open(list_file_name, 'w') as h : 
        for f in files:
            h.write(f"file '{f}' \n")
    txt=  f"ffmpeg -f concat -safe 0 -i {list_file_name} -c copy {os.path.join(workspace_dir, concat_file_name)}"
    pyperclip.copy(txt)
    return txt 


In [5]:
# gen_concat_cmd(left_video_dir, 'left_all_in_one.MP4')

In [6]:
# gen_concat_cmd(right_video_dir, 'right_all_in_one.MP4')

In [7]:
left_videos=fnmatch.filter(os.listdir(left_video_dir), f"*.{video_file_ext}")
right_videos=fnmatch.filter(os.listdir(right_video_dir), f"*.{video_file_ext}")
left_videos=sorted(left_videos, key=file_sort_key)
right_videos=sorted(right_videos, key=file_sort_key)

left_videos , right_videos

(['20260612204159_000014.MP4',
  '20260612205342_000015.MP4',
  '20260612210718_000016.MP4',
  '20260612212113_000017.MP4'],
 ['20260612204114_000017.MP4',
  '20260612205347_000018.MP4',
  '20260612210723_000019.MP4',
  '20260612212116_000020.MP4'])

In [8]:

# left_video_file=os.path.join(raw_video_dir, 'left','20260516181012_000004.MP4')
# right_video_file=os.path.join(raw_video_dir, 'right','r_20260516181403_000006.MP4')
# left_video_file=os.path.join(workspace_dir, 'left_all_in_one.MP4')
# right_video_file=os.path.join(workspace_dir, 'right_all_in_one.MP4')
quater=4

left_video_file=os.path.join(left_video_dir, left_videos[quater-1])
right_video_file=os.path.join(right_video_dir, right_videos[quater-1])

left_video_file, right_video_file

('/Users/feng/workspace/data/bball_video/left/20260612212113_000017.MP4',
 '/Users/feng/workspace/data/bball_video/right/20260612212116_000020.MP4')

In [9]:
from video.synchronizer import * 
raw_offset, meaning = find_offset_seconds(workspace_dir,left_video_file, right_video_file)
offset=round(raw_offset)
raw_offset, offset, meaning

/Users/feng/anaconda3/envs/vid_ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(-2.0916875, -2, 'video B started 2.0916875 earlier than video A.')

In [10]:
from video.ffmpeg_wrapper import * 
left_video_duration= get_video_duration(left_video_file)
right_video_duration= get_video_duration(right_video_file)
left_video_duration,right_video_duration 


(843.433333, 845.233333)

In [11]:
left_frames_dir=os.path.join(workspace_dir, 'left')
Path(left_frames_dir).mkdir(parents=True, exist_ok=True)

right_frames_dir=os.path.join(workspace_dir, 'right')
Path(right_frames_dir).mkdir(parents=True, exist_ok=True)

In [12]:
def get_start_end_time(offset, left_duration, right_duration): 
    if offset <0: 
        # right start early
        right_start=-1*offset
        new_right_duration =  right_duration+offset 
        min_duration =min(left_duration, new_right_duration)
        return 0, min_duration, right_start, right_start+min_duration

    elif offset>0: 
        left_start=offset 
        new_left_duration = left_duration-offset
        min_duration=min(new_left_duration, right_duration)
        return left_start, left_start+ min_duration, 0, min_duration
ts= get_start_end_time(offset, left_video_duration, right_video_duration)
ts =[format_seconds_to_hhmmss(t) for t in ts]
left_start, left_end, right_start, right_end= ts[0],ts[1],ts[2],ts[3]
left_start, left_end, right_start, right_end

('00:00:00', '00:14:03.233', '00:00:02', '00:14:05.233')

In [13]:
file_name_pattern=f"%d.{extract_img_fmt}"
output = os.path.join(left_frames_dir, file_name_pattern)
cmd=f"ffmpeg -ss {left_start} -to {left_end} -i {left_video_file} -vf scale=1280:720,fps={extract_fps} {output}"
stdout, stderr = call_command_line(cmd)
print(stdout)


In [14]:
output = os.path.join(right_frames_dir, file_name_pattern)
cmd=f"ffmpeg -ss {right_start} -to {right_end} -i {right_video_file} -vf scale=1280:720,fps={extract_fps} {output}"
stdout, stderr = call_command_line(cmd)
print(stdout)


In [15]:
def rename_img_file_to_seconds_index(fd):
    files = fnmatch.filter(os.listdir(fd), f"*.{extract_img_fmt}")
    for f in files: 
        parts=f.split(".")
        file_index, ext=int(parts[0])-1, parts[1]
        second, index_in_second=divmod(file_index,extract_fps)
        new_file_name=f"{second}-{index_in_second}.{ext}"
        shutil.move(os.path.join(fd,f), os.path.join(fd,new_file_name))
rename_img_file_to_seconds_index(left_frames_dir)
rename_img_file_to_seconds_index(right_frames_dir)

In [16]:
from ml.detectors import * 
import pandas  as pd 

detection_chunk_size=240

def detect_basketball_and_players(video_frame_dir): 
    files=fnmatch.filter(os.listdir(video_frame_dir), f"*.{extract_img_fmt}")
    df=pd.DataFrame(data=files, columns=["file_name"])
    df['chunk']=range(df.shape[0])
    df['chunk']=df.chunk//detection_chunk_size
    def detect_chunk(gdf):
        logging.info(f"object detection chuck : {gdf.name}")
        files=[os.path.join(video_frame_dir,f) for f in gdf.file_name] 
        results=detect_objects_on_multiple_images(files)
        trios=[analyze_result(r) for r in results]
        num_of_basketballs, num_of_players , num_of_possessions, total_object_area= zip (*trios)
        gdf['has_baseketball']=num_of_basketballs
        gdf['has_baseketball']=gdf.has_baseketball>0
        gdf['has_possession']=num_of_possessions
        gdf['has_possession']=gdf.has_possession>0
        gdf['total_num_of_players']=num_of_players
        gdf['total_object_area']=total_object_area
        
        return gdf 
    ret= df.groupby('chunk').apply(detect_chunk)
    ret.sort_values(by="file_name", inplace=True)
    ret.to_csv(os.path.join(video_frame_dir, "frame_information.csv"), index=False)
    # return pd.concat(dfs)
    return ret
detect_basketball_and_players(left_frames_dir).shape , detect_basketball_and_players(right_frames_dir).shape 


((2530, 5), (2530, 5))

In [17]:
def agg_frame_information_df(df):
    def extract_second_and_index(fn):
        parts = fn.split(".")[0].split("-") 
        return parts[0], parts[1]
    pairs=df.file_name.apply(extract_second_and_index)
    df['second'], df["index_within_second"]=zip (*pairs)
    df['second'], df["index_within_second"]=df.second.astype(int), df.index_within_second.astype(int)
    adf=df.groupby('second').agg(
        {
            "has_baseketball":'any',
            "total_num_of_players":'sum' ,
            "has_possession":'any', 
            "total_object_area":'sum'   
        } 
    ).reset_index()
    adf.sort_values(by='second', inplace=True)
    return adf 

    

left_frames_df=pd.read_csv(os.path.join(left_frames_dir, "frame_information.csv"), index_col=None)
left_frames_df=agg_frame_information_df(left_frames_df)

right_frames_df=pd.read_csv(os.path.join(right_frames_dir, "frame_information.csv"), index_col=None)
right_frames_df=agg_frame_information_df(right_frames_df)

left_frames_df.shape, right_frames_df.shape 


((844, 5), (844, 5))

In [18]:
left_frames_df.head(2)

,second,has_baseketball,total_num_of_players,has_possession,total_object_area
0,0,True,9,True,65241.197266
1,1,True,15,False,87825.923828


In [19]:
mdf=left_frames_df.merge(right_frames_df, how='inner', on='second', suffixes=("_l","_r"))
mdf.head(2) 

,second,has_baseketball_l,total_num_of_players_l,has_possession_l,total_object_area_l,has_baseketball_r,total_num_of_players_r,has_possession_r,total_object_area_r
0,0,True,9,True,65241.197266,False,6,False,32574.714355
1,1,True,15,False,87825.923828,False,2,False,7184.120361


In [20]:
def smooth_left_right(ac): 
    s=ac.copy()
    prev,nxt=s.shift(1),s.shift(-1)
    mask=(s!=prev) & (s!=nxt) & (prev==nxt)
    ret=s.copy()
    ret[mask]=prev[mask]

    #force setting the first to the same as second; 
    #the last to the same as the second last;
    if ret.size>2:
        ret.iloc[0]=ret.iloc[1]
        ret.iloc[-1]=ret.iloc[-2]

    return ret 

def decide_active_camera(r):
    # is_timeout_or_break=r['total_num_of_players_l']<=2 and r['total_num_of_players_r']<=2
    # if is_timeout_or_break: 
    #      return "skip"
    
    ball_on_left= r["has_possession_l"] or r["has_baseketball_l"]
    ball_on_right= r["has_possession_r"] or r["has_baseketball_r"]

    only_on_left=ball_on_left and not ball_on_right
    only_on_right=ball_on_right and not ball_on_left

    if only_on_left:
         return 'left'
    elif only_on_right: 
        return 'right'
    else: 
        if r['total_num_of_players_l']>r['total_num_of_players_r']:
                return 'left'
        elif r['total_num_of_players_l']<r['total_num_of_players_r']:
                return 'right'
        else:
            if r['total_object_area_l']>r['total_object_area_r']:
                    return 'left'
            elif r['total_object_area_l']<r['total_object_area_r']:
                    return 'right'
    return None

mdf.sort_values(by='second', ascending=True, inplace=True)

mdf['active_camera']=mdf.apply(decide_active_camera, axis=1)
mdf.ffill(inplace=True)

#double smooth 
mdf['active_camera']=smooth_left_right(mdf.active_camera)
mdf['active_camera']=smooth_left_right(mdf.active_camera)
mdf['active_camera']=smooth_left_right(mdf.active_camera)
mdf.to_csv(os.path.join(workspace_dir,'active_camera.csv'), index=False)
mdf.head(2)

,second,has_baseketball_l,total_num_of_players_l,has_possession_l,total_object_area_l,has_baseketball_r,total_num_of_players_r,has_possession_r,total_object_area_r,active_camera
0,0,True,9,True,65241.197266,False,6,False,32574.714355,left
1,1,True,15,False,87825.923828,False,2,False,7184.120361,left


In [21]:
mdf=pd.read_csv(os.path.join(workspace_dir,'active_camera.csv'), index_col=None)
def gen_hhmmss(side, offset, second): 
    #if offset <0 , then right start early; otherwise left start early. 
    if offset<0: 
        if side =='right': 
            second =float(second) +offset*(-1)
    else: 
        if side=='left': 
            second =float(second) +offset
    return format_seconds_to_hhmmss(second) 

def gen_ffmpeg_extract_commands(mdf):
    def gen_cmd(start_r,last_r): 
        if start_r.active_camera=='left':
            start_hhmmss=gen_hhmmss("left", offset, start_r.second)
            to_hhmmss=gen_hhmmss("left", offset, last_r.second)
            video_file_name=left_video_file
        else:
            start_hhmmss=gen_hhmmss("right", offset, start_r.second)
            to_hhmmss=gen_hhmmss("right", offset, last_r.second)
            video_file_name=right_video_file
        return [f"ffmpeg -ss {start_hhmmss} -to {to_hhmmss} -i {video_file_name}  -c copy part_to_be_replaced"]
    
    start_r, last_r=None, None  
    cmds=[]
    for r in mdf.itertuples():
        if start_r is None: 
            start_r, last_r=r, r 
            continue
        else: 
            if r.active_camera!=last_r.active_camera: 
                #need swtich. 
                cmds.extend(gen_cmd(start_r, last_r))
                start_r, last_r=r, r 
            else:
                last_r=r 

    if start_r and last_r: 
        cmds.extend(gen_cmd(start_r,last_r))       
    return cmds 
extract_cmds = gen_ffmpeg_extract_commands(mdf)
extract_cmds[-1]


'ffmpeg -ss 00:13:57 -to 00:14:03 -i /Users/feng/workspace/data/bball_video/left/20260612212113_000017.MP4  -c copy part_to_be_replaced'

In [22]:
import pyperclip
def post_process_extract_cmds(extract_cmds): 
    def replace(index, cmd): 
        part=os.path.join(workspace_dir, f"part-{index}.MP4")
        return cmd.replace('part_to_be_replaced', part)
    extract_cmds= [replace(i,c) for i, c in enumerate(extract_cmds) ]
    text="\n".join(extract_cmds)
    pyperclip.copy(text)

    #write to sh file. 
    with open(os.path.join(workspace_dir,'extract_part.sh'), 'w') as h: 
        h.write(text)
    
    return extract_cmds
    
extract_cmds=post_process_extract_cmds(extract_cmds)
! bash {workspace_dir}/extract_part.sh 

ffmpeg version 8.1.1 Copyright (c) 2000-2026 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.6.4.2)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.1.1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gpl --enable-libsvtav1 --enable-libopus --enable-libx264 --enable-libmp3lame --enable-libdav1d --enable-libvmaf --enable-libvpx --enable-libx265 --enable-openssl --enable-videotoolbox --enable-audiotoolbox --enable-neon
  libavutil      60. 26.101 / 60. 26.101
  libavcodec     62. 28.101 / 62. 28.101
  libavformat    62. 12.101 / 62. 12.101
  libavdevice    62.  3.101 / 62.  3.101
  libavfilter    11. 14.101 / 11. 14.101
  libswscale      9.  5.101 /  9.  5.101
  libswresample   6.  3.101 /  6.  3.101
Input #0, mov,mp4,m4a,3gp,3g2,mj2, from '/Users/feng/workspace/data/bball_video/left/20260612212113_000017.MP4':
  Metadata:
    major_brand     : mp42
    minor_version   : 0
    compatib

In [ ]:
def gen_concat_command(extract_cmds, output_file_name_path): 
    list_file_name=os.path.join(workspace_dir, 'part-list.txt')
    with open(list_file_name, 'w') as h: 
        for i in range(len(extract_cmds)):
            p=os.path.join(workspace_dir,f"part-{i}.MP4")  
            h.write(f"file '{p}' \n")
    txt=  f"ffmpeg -f concat -safe 0 -i {list_file_name} -c copy {output_file_name_path}"
    pyperclip.copy(txt)
    return txt 

output_path_file=os.path.join('/Users/feng/workspace/data/bball_video/output',f'20260612-vs-Drg-{quater}.MP4')
concat_cmd= gen_concat_command(extract_cmds, output_path_file)
! {concat_cmd}

ffmpeg version 8.1.1 Copyright (c) 2000-2026 the FFmpeg developers
  built with Apple clang version 17.0.0 (clang-1700.6.4.2)
  configuration: --prefix=/opt/homebrew/Cellar/ffmpeg/8.1.1 --enable-shared --enable-pthreads --enable-version3 --cc=clang --host-cflags= --host-ldflags= --enable-ffplay --enable-gpl --enable-libsvtav1 --enable-libopus --enable-libx264 --enable-libmp3lame --enable-libdav1d --enable-libvmaf --enable-libvpx --enable-libx265 --enable-openssl --enable-videotoolbox --enable-audiotoolbox --enable-neon
  libavutil      60. 26.101 / 60. 26.101
  libavcodec     62. 28.101 / 62. 28.101
  libavformat    62. 12.101 / 62. 12.101
  libavdevice    62.  3.101 / 62.  3.101
  libavfilter    11. 14.101 / 11. 14.101
  libswscale      9.  5.101 /  9.  5.101
  libswresample   6.  3.101 /  6.  3.101
[mov,mp4,m4a,3gp,3g2,mj2 @ 0x151615ae0] Auto-inserting h264_mp4toannexb bitstream filter
Input #0, concat, from '/Users/feng/workspace/runtime/tmp/aimpro/1637/part-list.txt':
  Duration: N

In [24]:
! rm -rf {workspace_dir}

In [ ]:
#compress. 
!ffmpeg -i {output_path_file} -c:v libx265 -crf 28 -preset slow -c:a aac -tag:v hvc1 -b:a 128k 265-{output_path_file}
